# PQ-TDAG — Publication Figure Notebook

Reads pre-computed data from `results/data/` and regenerates all figures.
**No liboqs, no NS-3 needed** — uses only the CSV/JSON outputs.

### How to use
1. Run **Cell 1** — upload `PQ-TDAG-main.zip`
2. Run **Cell 2** — install scipy
3. Run **Cell 3** — load all data
4. Run **Cell 4** — set style (edit colors/fonts here)
5. Run each figure cell
6. Run **last cell** — download all PDFs as zip

### Data files used
| File | Used by |
|------|---------|
| `results/data/crypto_timings.json` | All figures (scheme params) |
| `results/data/ns3_latency_cdf_*.csv` | fig_C1 (6 files, ~29k rows each) |
| `results/data/ns3_throughput_window_*.csv` | fig_B1 (4 files) |
| `results/data/ns3_throughput_scale_*.csv` | fig_B2 (4 files) |
| `results/data/ns3_erasure_M*.csv` | fig_C2 (4 files) |
| `results/data/byzantine_results.json` | fig_D1 |
| `results/data/gpu_benchmark.json` | fig_B3 |

## Cell 1 — Upload zip

In [ ]:
from google.colab import files
import zipfile, os
from pathlib import Path

uploaded = files.upload()           # select PQ-TDAG-main.zip

with zipfile.ZipFile('PQ-TDAG-main.zip','r') as z:
    z.extractall('.')

BASE = Path('PQ-TDAG-main')
DATA = BASE / 'results/data'
FIGS = BASE / 'results/figures'
FIGS.mkdir(parents=True, exist_ok=True)

print('Extracted. Data files:')
for f in sorted(DATA.iterdir()):
    print(f'  {f.name}')

## Cell 2 — Install

In [ ]:
!pip install scipy -q
print('OK')

## Cell 3 — Load all data

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats

# ── crypto timings (i9-14900KF, liboqs 0.15.0, 500 iter) ──────
with open(DATA/'crypto_timings.json') as f:
    raw = json.load(f)
S = raw['schemes']   # S['pq_tdag']['t_sign_mean_ms'] etc.

# ── NS-3 latency CDF (~29k rows per scheme) ────────────────────
LAT_IDS = ['pq_tdag','naive_mldsa44','falcon512','mldsa65','slhdsa128f','ecdsa']
latency = {}
for sid in LAT_IDS:
    p = DATA/f'ns3_latency_cdf_{sid}.csv'
    if p.exists():
        latency[sid] = pd.read_csv(p)

# ── NS-3 throughput ────────────────────────────────────────────
TP_IDS = ['pq_tdag','naive_mldsa44','falcon512','mldsa65']
tput_w = {s: pd.read_csv(DATA/f'ns3_throughput_window_{s}.csv')
           for s in TP_IDS if (DATA/f'ns3_throughput_window_{s}.csv').exists()}
tput_s = {s: pd.read_csv(DATA/f'ns3_throughput_scale_{s}.csv')
           for s in TP_IDS if (DATA/f'ns3_throughput_scale_{s}.csv').exists()}

# ── NS-3 erasure ───────────────────────────────────────────────
erasure = {M: pd.read_csv(DATA/f'ns3_erasure_M{M}.csv')
           for M in [3,5,8,10]
           if (DATA/f'ns3_erasure_M{M}.csv').exists()}

# ── Byzantine & GPU ────────────────────────────────────────────
with open(DATA/'byzantine_results.json') as f: BYZ = json.load(f)
with open(DATA/'gpu_benchmark.json')     as f: GPU = json.load(f)

print(f'Schemes loaded : {list(S.keys())}')
print(f'Latency CDFs   : {list(latency.keys())} ({len(next(iter(latency.values())))} rows each)')
print(f'Throughput     : {list(tput_w.keys())}')
print(f'Erasure M vals : {list(erasure.keys())}')

## Cell 4 — Style config (edit here)

In [ ]:
# ═══════════════════════════════════════════════════════
#  Edit anything below to change figure appearance
# ═══════════════════════════════════════════════════════

COLORS = dict(
    pq_tdag      = '#E63946',   # red   — proposed
    naive_mldsa44= '#F4A261',   # orange
    mldsa65      = '#457B9D',   # steel blue
    falcon512    = '#2A9D8F',   # teal
    slhdsa128s   = '#6A4C93',   # purple
    slhdsa128f   = '#9B59B6',   # violet
    xmssmt       = '#1D3557',   # dark navy
    ecdsa        = '#A8DADC',   # light cyan
)
MARKERS = dict(
    pq_tdag='o', naive_mldsa44='s', mldsa65='D',
    falcon512='^', slhdsa128s='v', slhdsa128f='P',
    xmssmt='X', ecdsa='h',
)

# Figure sizes (inches) — IEEE: single-col=3.5", double-col=7.0"
W1, W2, H = 3.5, 7.2, 4.5
LW_MAIN, LW_BASE, LW_REF = 2.5, 1.5, 1.2
MS_MAIN, MS_BASE         = 7, 5
SAVE_DPI                 = 300

plt.rcParams.update({
    'font.family'    : 'serif',
    'font.size'      : 11,
    'axes.labelsize' : 12,
    'axes.titlesize' : 11,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.grid'      : True,
    'grid.alpha'     : 0.35,
    'grid.linestyle' : '--',
    'figure.dpi'     : 110,
})

# Shortcuts
co = lambda s: COLORS.get(s,'#888')
mk = lambda s: MARKERS.get(s,'o')
lw = lambda s: LW_MAIN if s=='pq_tdag' else LW_BASE
ms = lambda s: MS_MAIN if s=='pq_tdag' else MS_BASE

# ICS constants
B_MAX, T_MAX, N_REF, F_REF, M_REF, SPL = 10.0, 50.0, 50, 20.0, 5, 100

def save(fig, name):
    for ext in ['pdf','png']:
        fig.savefig(FIGS/f'{name}.{ext}', bbox_inches='tight', dpi=SAVE_DPI)
    print(f'  Saved: {name}')
    plt.show()

def bw_naive(N,f,sig): return N*f*(SPL+sig)*8/1e6
def bw_pq(N,f,sig,M): return N*(f/M)*(M*SPL+sig)*8/1e6

print('Style applied.')

---
## fig_A1 — Bandwidth vs Frequency
`results/data/crypto_timings.json`

In [ ]:
FR  = np.array([1,2,4,5,8,10,15,20,30,40,50,60,80,100,120])
fig, ax = plt.subplots(figsize=(W1,H))

for sid in ['pq_tdag','naive_mldsa44','mldsa65','falcon512',
            'slhdsa128f','slhdsa128s','xmssmt','ecdsa']:
    if sid not in S: continue
    sig   = S[sid].get('sig_bytes',0)
    label = 'PQ-TDAG $M=5$ (Ours)' if sid=='pq_tdag' else S[sid].get('label',sid).replace(' [stateful]','').replace(' stateful','')
    bw    = [bw_pq(N_REF,f,sig,M_REF) for f in FR] if sid=='pq_tdag' \
            else [bw_naive(N_REF,f,sig) for f in FR]
    ax.plot(FR, bw, color=co(sid), marker=mk(sid),
            linewidth=lw(sid), markersize=ms(sid), label=label,
            zorder=5 if sid=='pq_tdag' else 2)

ax.axhline(B_MAX, color='black', linestyle='-.', linewidth=LW_REF,
           label=f'$B_{{\\max}}={B_MAX}$ Mbps')
ax.set(xlabel='Sampling Frequency $f$ (Hz)',
       ylabel='$B_{\\mathrm{req}}$ (Mbps)',
       title=f'Bandwidth vs. Frequency ($N={N_REF}$)',
       xlim=(FR[0],FR[-1]), ylim=(0,100))
ax.legend(fontsize=8.5, framealpha=0.9)
plt.tight_layout()
save(fig,'fig_A1_bw_vs_freq')

## fig_A2 — Bandwidth vs N
`results/data/crypto_timings.json`

In [ ]:
NR  = np.array([10,25,50,100,150,200,300,500,750,1000])
fig, ax = plt.subplots(figsize=(W1,H))

for sid in ['pq_tdag','naive_mldsa44','mldsa65','falcon512',
            'slhdsa128f','slhdsa128s','xmssmt','ecdsa']:
    if sid not in S: continue
    sig   = S[sid].get('sig_bytes',0)
    label = 'PQ-TDAG $M=5$ (Ours)' if sid=='pq_tdag' else S[sid].get('label',sid).replace(' [stateful]','').replace(' stateful','')
    bw    = [bw_pq(n,F_REF,sig,M_REF) for n in NR] if sid=='pq_tdag' \
            else [bw_naive(n,F_REF,sig) for n in NR]
    ax.plot(NR, bw, color=co(sid), marker=mk(sid),
            linewidth=lw(sid), markersize=ms(sid), label=label)

ax.axhline(B_MAX, color='black', linestyle='-.', linewidth=LW_REF,
           label=f'$B_{{\\max}}={B_MAX}$ Mbps')
ax.fill_between(NR, 0, B_MAX, alpha=0.06, color='#2A9D8F')
ax.set(xlabel='Number of Sensors $N$',
       ylabel='$B_{\\mathrm{req}}$ (Mbps)',
       title=f'Bandwidth vs. $N$ ($f={F_REF}$ Hz)',
       xlim=(NR[0],NR[-1]), ylim=(0,130))
ax.legend(fontsize=8.5, framealpha=0.9)
plt.tight_layout()
save(fig,'fig_A2_bw_vs_n')

## fig_A3 — M_min vs B_max
`results/data/crypto_timings.json`

In [ ]:
BR  = np.linspace(1,25,300)
fig, ax = plt.subplots(figsize=(W1,H))

for sid in ['pq_tdag','mldsa65','falcon512','slhdsa128f','slhdsa128s']:
    if sid not in S: continue
    sig   = S[sid].get('sig_bytes',0)
    label = 'PQ-TDAG (Ours)' if sid=='pq_tdag' else S[sid].get('label',sid)
    mm    = [min(np.ceil(sig/(b*1e6/(N_REF*F_REF*8)-SPL)),50)
             if (b*1e6/(N_REF*F_REF*8)-SPL)>0 else 50 for b in BR]
    ax.plot(BR, mm, color=co(sid), linewidth=lw(sid), label=label)

ax.axvline(B_MAX, color='black', linestyle='-.', linewidth=LW_REF,
           label=f'$B_{{\\max}}={B_MAX}$ Mbps')
ax.axhline(3, color='#888', linestyle=':', linewidth=0.9, alpha=0.7,
           label='$M_{\\min}=3$ reference')
ax.set(xlabel='Channel Capacity $B_{\\max}$ (Mbps)',
       ylabel='Minimum Window $M_{\\min}$',
       title='Corollary 1 — Feasibility Condition',
       xlim=(1,25), ylim=(0,30))
ax.legend(fontsize=9, framealpha=0.9)
plt.tight_layout()
save(fig,'fig_A3_mmin_vs_bmax')

## fig_A4 — W3: Falcon-512 scheme-agnostic comparison
`results/data/crypto_timings.json`

In [ ]:
M_vals = np.arange(1,11)
f_sig  = S['falcon512']['sig_bytes']
m_sig  = S['pq_tdag']['sig_bytes']
t_sf   = S['falcon512']['t_sign_mean_ms'];   t_sm = S['pq_tdag']['t_sign_mean_ms']
t_vf   = S['falcon512']['t_verify_mean_ms']; t_vm = S['pq_tdag']['t_verify_mean_ms']

E_BYTE, PROC = 0.0008, 4.0
E_sf = 26.6*(t_sf/12.0)/PROC; E_sm = 8.4*(t_sm/10.0)/PROC
E_vf = 0.07*(t_vf/0.032)/PROC; E_vm = 0.315*(t_vm/0.375)/PROC

N_test = np.arange(5,1001,5)
def maxN(sig,M):
    for n in reversed(N_test):
        if bw_pq(n,F_REF,sig,M)<=B_MAX: return n
    return 5

fig, axes = plt.subplots(1,3, figsize=(W2+1,H))
fig.suptitle('W3: Micro-Chaining is Scheme-Agnostic — Falcon-512 vs. ML-DSA-44',
             fontsize=10, y=1.01)

# (a) Bandwidth
bwf = [bw_pq(N_REF,F_REF,f_sig,M) for M in M_vals]
bwm = [bw_pq(N_REF,F_REF,m_sig,M) for M in M_vals]
axes[0].plot(M_vals,bwf,color=co('falcon512'),marker='^',lw=2.0,ms=6,label='Falcon-512')
axes[0].plot(M_vals,bwm,color=co('pq_tdag'), marker='o',lw=2.5,ms=7,label='ML-DSA-44 (Ours)')
axes[0].axhline(B_MAX,color='red',linestyle='-.',lw=1.2,label=f'$B_{{max}}={B_MAX}$')
bf2=bw_pq(N_REF,F_REF,f_sig,2); bm5=bw_pq(N_REF,F_REF,m_sig,5)
axes[0].annotate(f'Falcon M=2\n{bf2:.2f} Mbps',xy=(2,bf2),xytext=(4,bf2+2),fontsize=8,
    color=co('falcon512'),arrowprops=dict(arrowstyle='->',color=co('falcon512')))
axes[0].annotate(f'PQ-TDAG M=5\n{bm5:.2f} Mbps',xy=(5,bm5),xytext=(7,bm5+2),fontsize=8,
    color=co('pq_tdag'),arrowprops=dict(arrowstyle='->',color=co('pq_tdag')))
axes[0].set(xlabel='$M$',ylabel='$B_{req}$ (Mbps)',
    title='(a) Bandwidth — Falcon wins at M=2',xlim=(1,10),ylim=(0,25))
axes[0].legend(fontsize=8.5)

# (b) Energy
Ef = [E_sf/M+E_vf/M+E_BYTE*(SPL+f_sig/M) for M in M_vals]
Em = [E_sm/M+E_vm/M+E_BYTE*(SPL+m_sig/M) for M in M_vals]
ef2=E_sf/2+E_vf/2+E_BYTE*(SPL+f_sig/2)
em5=E_sm/5+E_vm/5+E_BYTE*(SPL+m_sig/5)
ratio=ef2/em5
axes[1].plot(M_vals,Ef,color=co('falcon512'),marker='^',lw=2.0,ms=6,label='Falcon-512')
axes[1].plot(M_vals,Em,color=co('pq_tdag'), marker='o',lw=2.5,ms=7,label='ML-DSA-44 (Ours)')
axes[1].annotate(f'Falcon M=2: {ef2:.2f}uJ\n({ratio:.1f}x more)',xy=(2,ef2),
    xytext=(4,ef2*1.05),fontsize=8,color=co('falcon512'),
    arrowprops=dict(arrowstyle='->',color=co('falcon512')))
axes[1].set(xlabel='$M$',ylabel='Energy/tx (uJ)',
    title=f'(b) Energy — ML-DSA wins ({ratio:.1f}x less)',xlim=(1,10))
axes[1].legend(fontsize=8.5)

# (c) Scalability
cf=[maxN(f_sig,M) for M in M_vals]
cm=[maxN(m_sig,M) for M in M_vals]
axes[2].plot(M_vals,cf,color=co('falcon512'),marker='^',lw=2.0,ms=6,label='Falcon-512')
axes[2].plot(M_vals,cm,color=co('pq_tdag'), marker='o',lw=2.5,ms=7,label='ML-DSA-44 (Ours)')
axes[2].axhline(1000,color='green',linestyle=':',lw=1.0,alpha=0.7)
axes[2].annotate(f'Falcon M=2: Nmax={cf[1]}',xy=(2,cf[1]),
    xytext=(4,cf[1]-70),fontsize=8,color=co('falcon512'),
    arrowprops=dict(arrowstyle='->',color=co('falcon512')))
axes[2].set(xlabel='$M$',ylabel='Max Feasible $N$',
    title='(c) Scalability — ML-DSA wins (10x more)',xlim=(1,10))
axes[2].legend(fontsize=8.5)

print(f'Falcon M=2 : {bf2:.2f}Mbps  {ef2:.2f}uJ  Nmax={cf[1]}')
print(f'PQ-TDAG M=5: {bm5:.2f}Mbps  {em5:.2f}uJ  Nmax={cm[4]}')
plt.tight_layout()
save(fig,'fig_A4_scheme_agnostic_comparison')

## fig_B1 — Throughput vs M
`results/data/ns3_throughput_window_*.csv`

In [ ]:
fig, ax = plt.subplots(figsize=(W1,H))
for sid in ['pq_tdag','naive_mldsa44','falcon512','mldsa65']:
    if sid not in tput_w: continue
    df    = tput_w[sid]
    label = 'PQ-TDAG (Ours)' if sid=='pq_tdag' else S[sid].get('label',sid)
    ax.plot(df['param'],df['throughput_tx_per_sec'],
            color=co(sid),marker=mk(sid),linewidth=lw(sid),markersize=ms(sid),
            linestyle='-' if sid=='pq_tdag' else '--',label=label)

ax.axvline(3,color='#888',linestyle=':',linewidth=1.2,label='$M_{min}=3$')
ax.set(xlabel='Window Size $M$',ylabel='$\\Gamma(\\mathcal{G})$ (tx/s)',
       title=f'Throughput vs. $M$ ($N={N_REF}$, $f={F_REF}$ Hz)',xlim=(1,25))
ax.legend(fontsize=9,framealpha=0.9)
plt.tight_layout()
save(fig,'fig_B1_throughput_vs_m')

## fig_B2 — Throughput vs N (scalability)
`results/data/ns3_throughput_scale_*.csv`

In [ ]:
fig, ax = plt.subplots(figsize=(W1,H))
for sid in ['pq_tdag','naive_mldsa44','falcon512','mldsa65']:
    if sid not in tput_s: continue
    df    = tput_s[sid]
    label = 'PQ-TDAG (Ours)' if sid=='pq_tdag' else S[sid].get('label',sid)
    ax.plot(df['param'],df['throughput_tx_per_sec'],
            color=co(sid),marker=mk(sid),linewidth=lw(sid),markersize=ms(sid),
            linestyle='-' if sid=='pq_tdag' else '--',label=label)

ax.annotate('Naive ML-DSA\ncollapse N=25',xy=(25,5),xytext=(80,300),
    fontsize=8.5,color=co('naive_mldsa44'),
    arrowprops=dict(arrowstyle='->',color=co('naive_mldsa44')))
ax.set(xlabel='Number of Sensors $N$',ylabel='Throughput (tx/s)',
       title=f'Scalability ($f={F_REF}$ Hz, $B_{{max}}={B_MAX}$ Mbps)')
ax.legend(fontsize=9,framealpha=0.9)
plt.tight_layout()
save(fig,'fig_B2_throughput_vs_n')

## fig_C1 — Latency CDF (most important)
`results/data/ns3_latency_cdf_*.csv` (~29k rows each)

In [ ]:
fig, ax = plt.subplots(figsize=(W1,H))

for sid in ['pq_tdag','naive_mldsa44','falcon512','mldsa65','slhdsa128f','ecdsa']:
    if sid not in latency: continue
    df    = latency[sid]
    label = 'PQ-TDAG (Ours)' if sid=='pq_tdag' else S.get(sid,{}).get('label',sid)
    # p99.99
    p9999 = df[df['cdf']<=0.9999]['latency_ms'].max()
    print(f'{sid:22s}  p99.99 = {p9999:.3f} ms   T_max/p99.99 = {T_MAX/p9999:.1f}x')
    ax.plot(df['latency_ms'],df['cdf'],
            color=co(sid),linewidth=lw(sid),
            linestyle='-' if sid=='pq_tdag' else '--',
            label=label)

ax.axvline(T_MAX,color='#E63946',linestyle='-.',linewidth=2.0,
           label=f'$T_{{max}}={T_MAX}$ ms')
ax.axvspan(T_MAX,70,alpha=0.07,color='#E63946')
ax.axhline(0.9999,color='#aaa',linestyle=':',linewidth=0.7,
           label='99.99th pctile')
ax.set(xlabel='Confirmation Latency (ms)',ylabel='CDF',
       title=f'Latency CDF ($N={N_REF}$, $f={F_REF}$ Hz, NS-3)',
       xlim=(0,60),ylim=(0,1.02))
ax.legend(fontsize=8.5,framealpha=0.9)
plt.tight_layout()
save(fig,'fig_C1_latency_cdf')

## fig_C2 — Latency vs erasure p_e
`results/data/ns3_erasure_M*.csv`

In [ ]:
ECOLS = [co('falcon512'),co('pq_tdag'),co('mldsa65'),co('slhdsa128s')]
fig, ax = plt.subplots(figsize=(W1,H))

for M, color in zip([3,5,8,10],ECOLS):
    if M not in erasure: continue
    df  = erasure[M]
    pe  = df['pe'].values*100
    lat = df['latency_ms_mean'].values
    std = df['latency_ms_std'].values
    ax.plot(pe,lat,color=color,linewidth=LW_BASE,label=f'$M={M}$')
    ax.fill_between(pe,lat-2*std,lat+2*std,alpha=0.15,color=color)

ax.axhline(T_MAX,color='#E63946',linestyle='-.',linewidth=LW_REF+0.5,
           label=f'$T_{{max}}={T_MAX}$ ms')
ax.set(xlabel='Erasure Probability $p_e$ (%)',
       ylabel='Latency (ms)',
       title='TBFR Resilience ($\\pm2\\sigma$ shaded)',
       xlim=(0,25),ylim=(0,T_MAX*1.8))
ax.legend(fontsize=9,framealpha=0.9)
plt.tight_layout()
save(fig,'fig_C2_latency_vs_pe')

## fig_sensitivity_tpipe — W2 sensitivity
Analytical (no data file needed)

In [ ]:
# Bug-fixed version: L_nominal not L_worst, m_max not m_min
t_sign = S['pq_tdag']['t_sign_mean_ms']
T_RTX, T_TIP = 3.0, 0.5

def L_nom(M,tp):  return t_sign + T_TIP + M*tp
def r_max(M,tp):
    sl = T_MAX - t_sign - T_TIP - M*tp
    return max(0,int(sl/T_RTX)) if sl>0 else 0
def safety(M,tp): return T_MAX/L_nom(M,tp) if L_nom(M,tp)>0 else 0

MR = np.arange(1,16)
TP = [1.0,3.0,5.0,10.0]
C4 = ['#E63946','#2A9D8F','#457B9D','#6A4C93']
L4 = ['-','--','-.',':']

fig, axes = plt.subplots(1,3,figsize=(W2+1,H))
fig.suptitle(f'W2 Sensitivity: $t_{{pipe}}$ sweep — '
             f'$t_{{sign}}={t_sign:.4f}$ ms, $T_{{max}}={T_MAX}$ ms',
             fontsize=10,y=1.01)

print(f'{"t_pipe":>10}  {"max_M":>8}  {"safety@M=5":>12}')
print('-'*35)
for tp,color,ls in zip(TP,C4,L4):
    lv = [L_nom(M,tp)  for M in MR]
    sv = [safety(M,tp) for M in MR]
    rv = [r_max(M,tp)  for M in MR]
    lab = f'$t_{{pipe}}={tp:.0f}$ ms'
    for ax,yv in zip(axes,[lv,sv,rv]):
        ax.plot(MR,yv,color=color,linestyle=ls,lw=2.0,marker='o',ms=4,label=lab)
    feasible = [M for M in MR if r_max(M,tp)>=1]
    m_max = max(feasible) if feasible else 0
    sm5   = safety(5,tp)
    print(f'{tp:>10.0f}  {m_max:>8}  {sm5:>10.1f}x')

sm_base = safety(5,1.0)
axes[0].axhline(T_MAX,color='red',linestyle='-.',lw=1.5,label=f'$T_{{max}}$')
axes[0].set(xlabel='$M$',ylabel='$L_{nom}$ (ms)',title='(a) Nominal Latency',
            xlim=(1,15),ylim=(0,T_MAX*1.6)); axes[0].legend(fontsize=8.5)

axes[1].axhline(1.0,color='red',linestyle='-.',lw=1.2)
axes[1].axhline(sm_base,color='green',linestyle=':',lw=1.2,alpha=0.8,
                label=f'{sm_base:.0f}x baseline')
axes[1].set(xlabel='$M$',ylabel='$T_{{max}}/L_{{nom}}$',
            title='(b) Safety Margin (nominal)',
            xlim=(1,15),ylim=(0,sm_base*1.3+2)); axes[1].legend(fontsize=8.5)

axes[2].axhline(0,color='red',linestyle='-.',lw=1.0)
axes[2].set(xlabel='$M$',ylabel='$r_{max}$',
            title='(c) TBFR Recovery Budget',
            xlim=(1,15)); axes[2].legend(fontsize=8.5)

plt.tight_layout()
save(fig,'fig_sensitivity_tpipe')

## fig_D1 — Byzantine robustness
`results/data/byzantine_results.json`

In [ ]:
ratios = [float(k) for k in BYZ]
integ  = [BYZ[k]['integrity_mean']*100 for k in BYZ]
detect = [BYZ[k]['equivoc_detect_ms']  for k in BYZ]
rp     = [r*100 for r in ratios]
bc     = ['#2A9D8F' if r<1/3 else '#E63946' for r in ratios]

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(W2,H-0.5))
ax1.bar(rp,integ,4.5,color=bc,edgecolor='black',linewidth=0.5)
ax1.axvline(33.3,color='#E63946',linestyle='--',lw=1.8,label='Threshold $n/3$')
ax1.set(xlabel='Byzantine Ratio (%)',ylabel='DAG Integrity (%)',
        title='(D1a) DAG Integrity ($n=9$)',xlim=(-1,36),ylim=(80,102))
ax1.legend(fontsize=9)

ax2.plot(rp,detect,color='#E63946',marker='o',lw=2.0,ms=7)
ax2.axvline(33.3,color='#E63946',linestyle='--',lw=1.5)
ax2.axhline(T_MAX,color='#888',linestyle='-.',lw=1.0,label=f'$T_{{max}}$')
ax2.set(xlabel='Byzantine Ratio (%)',ylabel='Detection Time (ms)',
        title='(D1b) Equivocation Detection',xlim=(-1,36))
ax2.legend(fontsize=9)
plt.tight_layout()
save(fig,'fig_D1_byzantine_robustness')

## fig_D4 — W6: Byzantine vs cluster size n
Analytical

In [ ]:
N_VALS = [3,5,7,9]
BYZ_R  = np.linspace(0,0.45,60)
G_COLS = ['#E63946','#F4A261','#2A9D8F','#457B9D']

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(W2,H-0.5))
fig.suptitle('W6: Byzantine Fault Tolerance vs. Gateway Cluster Size',
             fontsize=10,y=1.01)

print(f'{"n":>4}  {"threshold":>10}  {"mode":>10}')
for n, color in zip(N_VALS,G_COLS):
    thresh = (n-1)//3
    mode   = 'BFT' if thresh>0 else 'CFT-only'
    print(f'{n:>4}  {thresh:>10}  {mode:>10}')
    integ_n = [100.0 if r*n<thresh else
               max(60, 99-(r*n-thresh)*20) for r in BYZ_R]
    ax1.plot(BYZ_R*100,integ_n,color=color,lw=2.0,
             label=f'$n={n}$ (|F|max={thresh}, {mode})')
    if thresh>0:
        ax1.axvline(thresh/n*100,color=color,linestyle=':',lw=0.8,alpha=0.5)

ax1.set(xlabel='Byzantine Gateways (%)',ylabel='DAG Integrity (%)',
        title='(D4a) Integrity vs. Cluster Size',xlim=(0,45),ylim=(55,102))
ax1.legend(fontsize=8.5)

# Recommendation table
ax2.axis('off')
rows=[[str(n),str((n-1)//3),
       'BFT' if (n-1)//3>0 else 'CFT only',
       'Upgrade to n>=4' if n==3 else ('n>=7 for 2-fault' if (n-1)//3<2 else 'Recommended')]
      for n in N_VALS]
t=ax2.table(cellText=rows,colLabels=['n','|F|max','Mode','Recommendation'],
            cellLoc='center',loc='center',bbox=[0,0,1,1])
t.auto_set_font_size(False); t.set_fontsize(9)
for i,n in enumerate(N_VALS):
    thr=(n-1)//3
    c='#ffcccc' if thr==0 else ('#fff3cd' if thr<2 else '#d4edda')
    for j in range(4): t[i+1,j].set_facecolor(c)
ax2.set_title('(D4b) Deployment Recommendations',fontsize=10,pad=15)
plt.tight_layout()
save(fig,'fig_D4_byzantine_vs_n')

## fig_E2 — Energy per transaction (W1 corrected)
`results/data/crypto_timings.json`

In [ ]:
# W1 corrected: E_byte = P_tx * (8/R_tx) = 1mW * 8/10Mbps = 0.0008 uJ/byte
P_TX=1e-3; R_TX=10e6
E_BYTE = P_TX*(8.0/R_TX)*1e6   # = 0.0008 uJ/byte
assert abs(E_BYTE-0.0008)<1e-9
PROC=4.0

PQM4={'pq_tdag':(8.4,0.315,10),'naive_mldsa44':(8.4,0.315,10),
      'mldsa65':(14.7,0.49,16),'falcon512':(26.6,0.07,12),
      'slhdsa128s':(406,12.6,8500),'slhdsa128f':(22.4,24.5,415),
      'xmssmt':(101.5,8.4,1000),'ecdsa':(0.35,0.035,4.5)}

ORDER=['ecdsa','pq_tdag','naive_mldsa44','falcon512',
       'mldsa65','slhdsa128f','xmssmt','slhdsa128s']
xlabs,totals,bcolors=[],[],[]

for sid in ORDER:
    if sid not in S or 't_sign_mean_ms' not in S[sid]: continue
    s=S[sid]; M=M_REF if sid=='pq_tdag' else 1
    es,ev,tm=PQM4.get(sid,(8.4,0.315,10))
    tv_m=tm*(ev/max(es,1e-9))
    Es=es*(s['t_sign_mean_ms']/tm)/PROC
    Ev=ev*(s['t_verify_mean_ms']/max(tv_m,1e-9))/PROC
    Et=E_BYTE*(SPL+s.get('sig_bytes',2420)/M)
    total=Es/M+Ev/M+Et
    xlabs.append('PQ-TDAG\n(Ours)' if sid=='pq_tdag'
                 else s.get('label',sid).replace(' (FIPS','\n(FIPS').replace(' (Classical','\n(Classical').replace(' [stateful]','').replace(' stateful',''))
    totals.append(total); bcolors.append(co(sid))

fig,ax=plt.subplots(figsize=(W2+0.5,H))
x=np.arange(len(xlabs))
bars=ax.bar(x,totals,0.65,color=bcolors,edgecolor='black',linewidth=0.5)
our=next((i for i,s in enumerate(ORDER) if s=='pq_tdag'),None)
if our is not None:
    bars[our].set_edgecolor('#E63946'); bars[our].set_linewidth(2.2)
for b,v in zip(bars,totals):
    ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.3,
            f'{v:.1f}',ha='center',va='bottom',fontsize=8)
ax.text(0.01,0.97,
    f'$E_{{byte}}={E_BYTE}$ uJ/byte  ($P_{{tx}}=1$ mW, IEEE 802.15.4)',
    transform=ax.transAxes,fontsize=8,va='top',
    bbox=dict(boxstyle='round,pad=0.3',fc='lightyellow',ec='gray',alpha=0.9))
ax.set_xticks(x); ax.set_xticklabels(xlabs,fontsize=9)
ax.set(ylabel='Energy per Transaction (uJ)',
       title=f'Energy (W1 corrected, $M={M_REF}$ PQ-TDAG, $M=1$ others)',
       ylim=(0,max(totals)*1.18))
plt.tight_layout()
save(fig,'fig_E2_energy_per_tx')

## Download all figures

In [ ]:
import zipfile, os
from google.colab import files

pdfs = sorted([f for f in os.listdir(FIGS) if f.endswith('.pdf')])
print(f'Packaging {len(pdfs)} figures:')
for f in pdfs: print(f'  {f}')

with zipfile.ZipFile('pq_tdag_figures.zip','w') as zf:
    for f in pdfs:
        zf.write(FIGS/f, f)

files.download('pq_tdag_figures.zip')
print('Done.')